In [1]:
# Mount Drive + Imports + Data Loading

# ================== INSTALL ==================
!pip install -q transformers datasets scikit-learn accelerate

# ================== MOUNT DRIVE ==================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ================== IMPORTS ==================
import os, json, torch
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

# ================== LOAD DATA ==================
path = "/content/drive/MyDrive/colab_datasets/dataset.json"

if not os.path.exists(path):
    raise FileNotFoundError("Dataset not found. Check path!")

with open(path, "r") as f:
    data = json.load(f)

texts, labels = [], []

for key in data:
    entry = data[key]
    text = " ".join(entry["post_tokens"])

    # Majority voting
    ann_labels = [ann["label"] for ann in entry["annotators"]]
    final_label = max(set(ann_labels), key=ann_labels.count)

    texts.append(text)
    labels.append(final_label)

df = pd.DataFrame({"text": texts, "label": labels})

print("Sample:\n", df.head())

# ================== ENCODE LABELS ==================
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])
num_labels = len(le.classes_)

# ================== SPLIT ==================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"], df["label"], test_size=0.2, stratify=df["label"], random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

print("Train:", len(train_texts), "Val:", len(val_texts), "Test:", len(test_texts))

# ================== METRICS FUNCTION ==================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="macro"),
        "recall": recall_score(labels, preds, average="macro"),
        "f1": f1_score(labels, preds, average="macro")
    }

Mounted at /content/drive
Sample:
                                                 text       label
0  i dont think im getting my baby them white 9 h...      normal
1  we cannot continue calling ourselves feminists...      normal
2                      nawt yall niggers ignoring me      normal
3  <user> i am bit confused coz chinese ppl can n...  hatespeech
4  this bitch in whataburger eating a burger with...  hatespeech
Train: 16118 Val: 2015 Test: 2015


In [ ]:
# BERT

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

# ================== TOKENIZER ==================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(texts):
    return tokenizer(list(texts), padding=True, truncation=True, max_length=128)

train_enc = tokenize(train_texts)
val_enc   = tokenize(val_texts)
test_enc  = tokenize(test_texts)

# ================== DATASET ==================
class HateDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HateDataset(train_enc, train_labels)
val_dataset   = HateDataset(val_enc, val_labels)
test_dataset  = HateDataset(test_enc, test_labels)

# ================== MODEL ==================
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=num_labels
)

# ================== TRAIN ==================
training_args = TrainingArguments(
    output_dir="./bert_base",
    learning_rate=2e-5,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(2)]
)

trainer.train()

# ================== RESULTS ==================
print("Validation:", trainer.evaluate())

print("\nTrain Performance:")
train_preds = trainer.predict(train_dataset)
print(classification_report(train_labels, np.argmax(train_preds.predictions, axis=1)))

print("\nTest Performance:")
test_preds = trainer.predict(test_dataset)
print(classification_report(test_labels, np.argmax(test_preds.predictions, axis=1)))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Validation: {'eval_loss': 0.7224624752998352, 'eval_accuracy': 0.6908188585607941, 'eval_precision': 0.6788186581900186, 'eval_recall': 0.6715858291727833, 'eval_f1': 0.6743143147728738, 'eval_runtime': 11.6149, 'eval_samples_per_second': 173.484, 'eval_steps_per_second': 21.696, 'epoch': 4.0}

Train Performance:
              precision    recall  f1-score   support

           0       0.88      0.87      0.87      4748
           1       0.85      0.91      0.88      6986
           2       0.77      0.70      0.73      4384

    accuracy                           0.84     16118
   macro avg       0.83      0.83      0.83     16118
weighted avg       0.84      0.84      0.84     16118


Test Performance:


              precision    recall  f1-score   support

           0       0.78      0.74      0.76       594
           1       0.72      0.75      0.74       873
           2       0.51      0.51      0.51       548

    accuracy                           0.68      2015
   macro avg       0.67      0.67      0.67      2015
weighted avg       0.68      0.68      0.68      2015



In [ ]:
# Bert and Class Weights

from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from torch import nn

model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_fn(texts):
    return tokenizer(list(texts), truncation=True, max_length=128)

# Dataset
class HateDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenize_fn(texts)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HateDataset(train_texts, train_labels)
val_dataset   = HateDataset(val_texts, val_labels)
test_dataset  = HateDataset(test_texts, test_labels)

# Class weights
weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float).to("cuda")

# Custom Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits.view(-1, num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=num_labels)

training_args = TrainingArguments(
    output_dir="./bert_weighted",
    learning_rate=2e-5,
    num_train_epochs=10,
    per_device_train_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

# ================== RESULTS ==================
print("\nValidation Metrics:")
print(trainer.evaluate())

print("\nTrain Report:")
train_preds = trainer.predict(train_dataset)
print(classification_report(train_labels, np.argmax(train_preds.predictions, axis=1)))

print("\nTest Report:")
test_preds = trainer.predict(test_dataset)
print(classification_report(test_labels, np.argmax(test_preds.predictions, axis=1)))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.836989,0.773229,0.625806,0.657663,0.651403,0.629698
2,0.671704,0.758672,0.674442,0.685459,0.674171,0.673458
3,0.531208,0.842437,0.659057,0.647650,0.653056,0.649709
4,0.377729,1.005193,0.663027,0.648810,0.654154,0.651084
5,0.254276,1.137341,0.641191,0.635094,0.643037,0.634735
6,0.176813,1.414277,0.650124,0.622842,0.632328,0.621142
7,0.124036,1.523075,0.658561,0.639646,0.644503,0.640893
8,0.091760,1.717636,0.657072,0.641115,0.645317,0.642852
9,0.065901,1.856592,0.644169,0.635629,0.637320,0.635655
10,0.053014,1.916819,0.646650,0.631582,0.637761,0.634105


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Validation Metrics:


{'eval_loss': 0.7597153782844543, 'eval_accuracy': 0.6764267990074442, 'eval_precision': 0.6861365000292098, 'eval_recall': 0.6752425294758462, 'eval_f1': 0.6748444857366801, 'eval_runtime': 4.5488, 'eval_samples_per_second': 442.97, 'eval_steps_per_second': 55.399, 'epoch': 10.0}

Train Report:
              precision    recall  f1-score   support

           0       0.88      0.81      0.84      4748
           1       0.89      0.80      0.84      6986
           2       0.64      0.81      0.72      4384

    accuracy                           0.80     16118
   macro avg       0.80      0.80      0.80     16118
weighted avg       0.82      0.80      0.81     16118


Test Report:


              precision    recall  f1-score   support

           0       0.81      0.71      0.75       594
           1       0.77      0.66      0.71       873
           2       0.47      0.64      0.54       548

    accuracy                           0.67      2015
   macro avg       0.68      0.67      0.67      2015
weighted avg       0.70      0.67      0.68      2015



In [ ]:
# XLM-RoBERTa

# ================== XLM-R ==================
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
model_ckpt = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# ================== DATASET ==================
class HateDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenize_fn(texts)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HateDataset(train_texts, train_labels)
val_dataset   = HateDataset(val_texts, val_labels)
test_dataset  = HateDataset(test_texts, test_labels)

# ================== MODEL ==================

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt, num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir="./xlm_results",
    learning_rate=2e-5,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

# ================== RESULTS ==================
print("\nValidation Metrics:")
print(trainer.evaluate())

print("\nTrain Report:")
train_preds = trainer.predict(train_dataset)
print(classification_report(train_labels, np.argmax(train_preds.predictions, axis=1)))

print("\nTest Report:")
test_preds = trainer.predict(test_dataset)
print(classification_report(test_labels, np.argmax(test_preds.predictions, axis=1)))

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Validation Metrics:


{'eval_loss': 0.7450054287910461, 'eval_accuracy': 0.6774193548387096, 'eval_precision': 0.6615521776261727, 'eval_recall': 0.6560543663483522, 'eval_f1': 0.6551254890773882, 'eval_runtime': 4.5613, 'eval_samples_per_second': 441.756, 'eval_steps_per_second': 55.247, 'epoch': 10.0}

Train Report:
              precision    recall  f1-score   support

           0       0.78      0.81      0.80      4748
           1       0.74      0.81      0.77      6986
           2       0.60      0.48      0.54      4384

    accuracy                           0.72     16118
   macro avg       0.71      0.70      0.70     16118
weighted avg       0.71      0.72      0.71     16118


Test Report:


              precision    recall  f1-score   support

           0       0.75      0.77      0.76       594
           1       0.70      0.76      0.73       873
           2       0.55      0.45      0.49       548

    accuracy                           0.68      2015
   macro avg       0.67      0.66      0.66      2015
weighted avg       0.67      0.68      0.67      2015



In [2]:
# T5 Model

from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

tokenizer = T5Tokenizer.from_pretrained("t5-small")

class T5Dataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.inputs = tokenizer(
            ["classify: " + str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=256
        )
        self.targets = tokenizer(
            list(map(str, labels)),
            padding="max_length",
            truncation=True,
            max_length=10
        )

    def __getitem__(self, idx):
        labels = torch.tensor(self.targets["input_ids"][idx])
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids": torch.tensor(self.inputs["input_ids"][idx]),
            "attention_mask": torch.tensor(self.inputs["attention_mask"][idx]),
            "labels": labels
        }

    def __len__(self):
        return len(self.inputs["input_ids"])

train_dataset = T5Dataset(train_texts.tolist(), train_labels.tolist())
val_dataset   = T5Dataset(val_texts.tolist(), val_labels.tolist())
test_dataset  = T5Dataset(test_texts.tolist(), test_labels.tolist())

model = T5ForConditionalGeneration.from_pretrained("t5-small")

training_args = TrainingArguments(
    output_dir="./t5_results",
    learning_rate=3e-5,
    num_train_epochs=10,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

# ================== PREDICTION FUNCTION ==================
def get_preds(dataset):
    model.eval()
    preds = []

    for batch in dataset:
        input_ids = batch["input_ids"].unsqueeze(0).to(model.device)
        attention_mask = batch["attention_mask"].unsqueeze(0).to(model.device)

        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=5)
        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

        preds.append(int(pred) if pred.isdigit() else -1)

    return preds

# ================== RESULTS ==================
test_preds = get_preds(test_dataset)
test_true = test_labels.tolist()

print("\nTest Classification Report:")
print(classification_report(test_true, test_preds))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Test Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.78      0.75       594
           1       0.69      0.73      0.71       873
           2       0.54      0.44      0.48       548

    accuracy                           0.67      2015
   macro avg       0.65      0.65      0.65      2015
weighted avg       0.66      0.67      0.66      2015



In [3]:
from sklearn.metrics import classification_report, accuracy_score

# ================== TRAIN RESULTS ==================
train_preds = get_preds(train_dataset)
train_true = train_labels.tolist()

print("\nTrain Classification Report:")
print(classification_report(train_true, train_preds))
print("Train Accuracy:", accuracy_score(train_true, train_preds))


# ================== VALIDATION RESULTS ==================
val_preds = get_preds(val_dataset)
val_true = val_labels.tolist()

print("\nValidation Classification Report:")
print(classification_report(val_true, val_preds))
print("Validation Accuracy:", accuracy_score(val_true, val_preds))


Train Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.84      0.79      4748
           1       0.74      0.80      0.77      6986
           2       0.62      0.47      0.53      4384

    accuracy                           0.72     16118
   macro avg       0.70      0.70      0.70     16118
weighted avg       0.71      0.72      0.71     16118

Train Accuracy: 0.7202506514455888

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.78      0.74       593
           1       0.70      0.77      0.73       874
           2       0.56      0.41      0.47       548

    accuracy                           0.67      2015
   macro avg       0.65      0.65      0.65      2015
weighted avg       0.66      0.67      0.66      2015

Validation Accuracy: 0.6744416873449132
